# Leitura de Parquet com PySpark

Notebook simples para ler e explorar arquivos Parquet usando Apache Spark.

## 1. Configurar SparkSession

Importar PySpark e criar uma SparkSession.

In [ ]:
from pyspark.sql import SparkSession

# Criar SparkSession
spark = SparkSession.builder \
    .appName("Leitura Parquet") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
spark

## 2. Ler Arquivo Parquet

Carregar um arquivo Parquet em um DataFrame Spark. Altere o caminho abaixo para o seu arquivo.

In [ ]:
# Altere o caminho para o seu arquivo Parquet
caminho_parquet = "quickbooks_data/daily_incremental/Account_20260207170553.parquet"

df = spark.read.parquet(caminho_parquet)
print(f"Arquivo carregado com sucesso! Registros: {df.count()}")

## 3. Transformar JSON em Tabela

A coluna do Parquet contém dados em formato JSON. Vamos parsear o JSON e expandir em colunas.

In [ ]:
from pyspark.sql.functions import from_json, schema_of_json, col

# Identificar o nome da coluna com JSON
col_json = df.columns[0]
print(f"Coluna JSON: {col_json}")

# Pegar uma amostra para inferir o schema
amostra = df.select(col_json).first()[0]
json_schema = schema_of_json(amostra)

# Parsear o JSON e expandir em colunas
df_parsed = df.withColumn("parsed", from_json(col(col_json), json_schema)) \
              .select("parsed.*")

print(f"Colunas resultantes: {df_parsed.columns}")
df_parsed.printSchema()

In [ ]:
# Visualizar os dados parseados
df_parsed.show(truncate=False)

In [ ]:
# Estatísticas descritivas
df_parsed.describe().show()

## 4. Executar Consultas SQL no Parquet

Registrar o DataFrame como view temporária e executar SQL.

In [ ]:
# Registrar como view temporária
df_parsed.createOrReplaceTempView("dados")

# Executar consulta SQL
resultado = spark.sql("SELECT * FROM dados LIMIT 10")
resultado.show()

In [ ]:
# Encerrar SparkSession
spark.stop()
print("SparkSession encerrada.")